# Event Waveform Processing and Analysis

This notebook processes event waveforms from the Cascadia dataset:
1. Reads event and pick information from CSV files
2. Downloads waveforms from IRIS FDSN
3. Processes waveforms (resampling, filtering)
4. Calculates amplitudes for local magnitude estimation
5. Visualizes waveforms with picks

by Marine Denolle (mdenolle@uw.edu)

In [1]:
# Import required libraries
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import obspy
from obspy import UTCDateTime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from utils.data_client import get_waveforms


In [ ]:
# pnwstore is installed from source — run once if not already installed:
# git clone https://github.com/niyiyu/pnwstore.git && cd pnwstore && pip install .

ERROR: Could not find a version that satisfies the requirement pnwstore (from versions: none)
ERROR: No matching distribution found for pnwstore


In [ ]:
# Set processing parameters
sample_rate = 100  # Hz
highpass_freq = 2  # Hz
window_after = 120  # seconds after origin time
window_before = 30  # seconds before origin time
P_before = 1
P_after = 2

# Data source: 'fdsn' (public EarthScope/IRIS) or 'pnwstore' (UW internal archive)
source = 'pnwstore'


/home/mdenolle/cascadia_obs_ensemble/.venv/lib64/python3.11/site-packages/obspy/clients/fdsn/client.py:251: ObsPyDeprecationWarning: IRIS is now EarthScope, please consider changing the FDSN client short URL to 'EARTHSCOPE'.
  warnings.warn(msg, ObsPyDeprecationWarning)
PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09


In [ ]:
#   try:
#         if network in ['NC', 'BK']:
#             # Query waveforms
#             _sdata = client_ncedc.get_waveforms(network=network, station=station, location="*", channel=channels,
#                                                starttime=UTCDateTime(t1), endtime=UTCDateTime(t2))
#         else:
#             # Shouldn't this have an explicit starttime + endtime inputs?
#             _sdata = client_waveform.get_waveforms(network=network, station=station, channel=channels, 
#                                               year=t1.strftime('%Y'), month=t1.strftime('%m'), 
#                                               day=t1.strftime('%d'))
#     except obspy.clients.fdsn.header.FDSNNoDataException:
#         Logger.warning(f"WARNING: No data for {network}.{station}.{channels} on {t1} - {t2}.")
#         return
    
#     # Create a new stream
#     sdata = Stream()
#     # Check if loaded data have a vertical component (minimum requirement)
#     has_Z = bool(_sdata.select(channel='??Z'))
#     # Check for HH and BH channels presence
#     has_HH = bool(_sdata.select(channel="HH?"))
#     has_BH = bool(_sdata.select(channel="BH?"))
#     has_EH = bool(_sdata.select(channel="EH?"))
# #     has_EN = bool(_sdata.select(channel="EN?"))

# hoose ~1 second before to 2 second after the pick tim
# 4 Hz highpass - this might be ok, and maybe preferable for OBS, but normally I would use more like ~2 Hz.

In [5]:
# Read event and pick data
events_df = pd.read_csv('../data/Cascadia_relocated_catalog_ver_3.csv')
picks_df = pd.read_csv('../data/Cascadia_relocated_catalog_picks_ver_3.csv')

print("Event data summary:")
print(f"Number of events: {len(events_df)}")
print("\nFirst few events:")
print(events_df.head())

print("\nPick data summary:")
print(f"Number of picks: {len(picks_df)}")
print("\nFirst few picks:")
print(picks_df.head())

Event data summary:
Number of events: 63887

First few events:
   Latitude   Longitude   Depth (km)             Origin Time (UTC)  \
0  47.22533  -122.16895       56.111   2010-01-01T00:15:17.262282Z   
1  48.19518  -121.77276        3.820   2010-01-01T00:16:49.375360Z   
2  47.86208  -122.09903       17.799   2010-01-01T07:18:03.689209Z   
3  47.96435  -122.91906       21.286   2010-01-01T08:51:56.371091Z   
4  45.87262  -122.19180        9.822   2010-01-01T16:12:43.838660Z   

    Uncertainity (km)   Horizontal Uncertainity (km)   Geometric Std. (km)  \
0              10.223                         10.216                 0.790   
1               7.560                          3.786                 0.140   
2               5.118                          4.807                 0.195   
3               1.899                          1.884                 0.287   
4               2.842                          2.838                 0.229   

    Detection Value   Num. P   Num. S   RMS Res

In [ ]:
def process_event(event_id, events_df, picks_df, source='fdsn'):
    """
    Process single event and return waveforms and metadata
    
    Parameters:
    -----------
    event_id : str
        Event identifier
    events_df : pandas.DataFrame
        DataFrame containing event information
    picks_df : pandas.DataFrame
        DataFrame containing pick information
    source : str
        Waveform backend: 'fdsn' (public EarthScope) or 'pnwstore' (UW internal archive)
    
    Returns:
    --------
    st : obspy.Stream
        Processed waveforms
    amplitudes : dict
        Peak amplitudes for each station
    origin_time : UTCDateTime
        Event origin time
    """

    # window_before and window_after are defined globally
    global window_before, window_after
    # Get event information
    event_cols = [col for col in events_df.columns if 'event' in col.lower()][0]
    origin_cols = [col for col in events_df.columns if 'origin' in col.lower()][0]
    station_cols = [col for col in picks_df.columns if 'station' in col.lower()][0]
    event = events_df[events_df[event_cols] == event_id].iloc[0]
    origin_time = UTCDateTime(event[origin_cols])
    
    # Get associated picks
    event_picks = picks_df[picks_df[event_cols] == event_id]
    # Storage for waveforms and amplitudes
    st = obspy.Stream()
    station_amplitudes = {}
    
    # Process each station's data
    for _, pick in event_picks.iterrows():
        try:
            # Get network and station codes (picks stored as "station.network")
            station, network = pick[station_cols].split('.')
            station = station.strip()
            network = network.strip()

            # Download waveforms — routes to pnwstore (UW/internal) or FDSN (NC/BK/public)
            st_temp = get_waveforms(
                network=network,
                station=station,
                channel="*H*",
                starttime=origin_time - window_before,
                endtime=origin_time + window_after,
                source=source,
            )
            
            # Process each trace
            for tr in st_temp:
                tr.resample(sample_rate)
                tr.filter('highpass', freq=highpass_freq)
                st += tr
                
            # Calculate amplitude (average of 3 components)
            station_amps = []
            for comp in ['Z', 'N', 'E']:
                comp_tr = st_temp.select(component=comp)
                comp_tr.trim(starttime=origin_time - P_before, endtime=origin_time + P_after)
                if len(comp_tr) > 0:
                    station_amps.append(np.max(np.abs(comp_tr[0].data)))
            
            if station_amps:
                station_amplitudes[network + '.' + station] = np.mean(station_amps)
            
        except Exception as e:
            print(f"Error processing {network}.{station}: {e}")
            continue
    
    return st, station_amplitudes, origin_time


In [ ]:
event_id = events_df[' Event ID '].values.tolist()
st, amplitudes, origin_time = process_event(event_id[0], events_df, picks_df, source=source)


In [11]:
picks_df.keys()

Index(['Pick Time (UTC)', ' Station Name', ' Phase Type', ' Residual (s)',
       ' Event ID ', ' Pick ID '],
      dtype='str')

In [ ]:
# for each event, process the waveforms and store amplitudes in a new column in picks_df
for eid in event_id:
    st, amplitudes, origin_time = process_event(eid, events_df, picks_df, source=source)
    # Store amplitudes in picks_df
    for station_key, amp in amplitudes.items():
        network, station = station_key.split('.')
        # find the row in picks_df based on any unique identifier in Station regardless of empty spaces
        station_col = picks_df[' Station Name'].str.strip()
        # find the row in picks_df based on Event ID and Station
        picks_df.loc[(picks_df[' Event ID '] == eid) & (station_col == station+"."+network), ' Amplitude '] = amp
    #print updated picks_df
    print(picks_df[picks_df[' Event ID '] == eid])

# save picks_df with amplitudes to a new csv file
picks_df.to_csv('../data/Cascadia_relocated_catalog_picks_with_amplitudes_ver_3.csv', index=False)


               Pick Time (UTC)  Station Name   Phase Type   Residual (s)  \
0  2010-01-01T00:15:27.180000Z       PCMD.UW            0          0.049   
1  2010-01-01T00:15:37.840400Z        RVW.UW            0          1.264   
2  2010-01-01T00:15:33.280000Z       PCMD.UW            1         -0.243   
3  2010-01-01T00:15:42.002000Z        GNW.UW            1          2.402   
4  2010-01-01T00:15:43.618400Z       B013.PB            1         -0.651   
5  2010-01-01T00:15:43.768400Z       B943.PB            1         -0.511   
6  2010-01-01T00:15:48.060400Z        BOW.UW            1         -0.263   

    Event ID    Pick ID    Amplitude   
0           0          0   431.549179  
1           0          1    29.641007  
2           0          2   431.549179  
3           0          3  1188.097291  
4           0          4     8.536636  
5           0          5     7.387047  
6           0          6    12.776827  


KeyboardInterrupt: 